# v6 architecture on the original UNSW-NB15 — does the plateau follow the traffic or the extractor?

The v6 pipeline (`final_model_v6.ipynb`, macro-F1 0.6485 / accuracy 0.9123), re-run end to end on
**`Resource/UNSW-NB15/`** — the original Moustafa & Slay (2015) dataset — instead of
`CIC-UNSW-Data.csv`.

## What this experiment is, and what it is not

**It is not a transfer experiment.** The two files share **zero columns**. CICFlowMeter emits 76
timing/size statistics; UNSW-NB15 emits 42 features of a different design. There is no shared space
in which a model fitted on one can be evaluated on the other, so "attach the dataset" can only mean
*port the architecture*, not *transfer the model*.

**It is not a fresh-traffic generalisation test either.** Per the project documentation,
`CIC-UNSW-NB15-2024` is the **UNSW-NB15 attack taxonomy re-extracted** with CICFlowMeter. Same source
traffic, same ten families, same names — two different feature extractors. So this does **not** close
the "single-dataset evaluation" gap in the usual sense (that needs different traffic, e.g.
CIC-IDS2017), and it should not be written up as if it did.

**What it is: a controlled test of the project's central claim.** Every negative result in this
project was eventually explained by one mechanism —

> the plateau at ~0.63 is a property of the **CICFlowMeter representation**, not of the task
> (`knn_class_overlap.ipynb`, `label_ambiguity_ceiling.ipynb`).

If that is right, holding the traffic and the taxonomy fixed while swapping the extractor should move
the result. If the same families fail in the same way on a completely different feature design, the
claim is much weaker and the difficulty is in the traffic.

**The design makes this the sharpest available test**, because UNSW-NB15 contains exactly the two
things this project found it needed and could not build:

| What the project concluded it needed | Available in UNSW-NB15? |
|---|---|
| **per-host / per-window context** — `knn_class_overlap.ipynb` named this as the missing ingredient; `feature_engineering_v2.ipynb` proved it is *impossible* on the CIC file (no IP, port or timestamp column) | **yes** — `ct_srv_src`, `ct_dst_ltm`, `ct_src_dport_ltm`, `ct_dst_src_ltm`, `ct_src_ltm`, `ct_srv_dst`, `ct_state_ttl` are all connection-history aggregates |
| **scale-free / normalised flow quantities** — 31 of these were hand-engineered in round 1 for a +0.0101 gain | **already native** — `rate`, `sload`, `dload`, `smean`, `dmean`, `sinpkt`, `dinpkt`, `sjit`, `djit` |

So UNSW-NB15 is not merely "a second dataset". It is the CIC representation's missing half, on the
same traffic.

## Protocol

Identical in structure to `final_model_v6.ipynb`, and deliberately so:

dedupe -> zero-variance drop -> $\lvert r\rvert \ge 0.95$ prune -> stratified 80/20 -> derived
features (label-free) -> Stage-A gate + Stage-B 9-family pool of 7 alpha-diverse members ->
5-fold cross-fit -> per-class $W$ on COMB -> decision-space dynamic combiner -> SELV selection ->
test touched once.

Two dataset-specific steps, both label-free and both before the split:

- **`id` and `label` are dropped.** `label` is the binary attack flag — keeping it would hand the
  Stage-A gate its own target.
- **`proto` (133), `service` (13), `state` (11) are encoded** — one-hot for the two small ones,
  ordinal codes for `proto`.

> **Runtime ~15 min.** Fewer rows than CIC after dedupe, so this is cheaper than v6.

## 1. Data — load, encode, dedupe, prune, split

In [1]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS, EPS = 42, 1e-6

D = "Resource/UNSW-NB15/"
raw = pd.concat([pd.read_csv(D + "UNSW_NB15_training-set.csv"),
                 pd.read_csv(D + "UNSW_NB15_testing-set.csv")], ignore_index=True)
raw.columns = [c.strip().lstrip("\ufeff") for c in raw.columns]
print("pooled the official train/test files: %d rows x %d cols" % raw.shape)

# --- label: attack_cat -> the SAME 0-9 taxonomy the CIC notebooks use ---
CAT = {"Normal":0, "Analysis":1, "Backdoor":2, "DoS":3, "Exploits":4,
       "Fuzzers":5, "Generic":6, "Reconnaissance":7, "Shellcode":8, "Worms":9}
assert set(raw.attack_cat.unique()) == set(CAT), set(raw.attack_cat.unique()) ^ set(CAT)
y_all = raw.attack_cat.map(CAT).astype(int)

# --- drop identifiers and the leaking binary flag ---
#     `label` IS the Stage-A target; leaving it in would hand the gate a perfect feature.
X_all = raw.drop(columns=["id", "attack_cat", "label"])
print("dropped: id, attack_cat (-> y), label (binary attack flag - would leak the gate's target)")

# --- categorical encoding: label-free, so it may precede the split ---
CATCOLS = ["proto", "service", "state"]
print("cardinality:", {c: X_all[c].nunique() for c in CATCOLS})
oh = pd.get_dummies(X_all[["service", "state"]], prefix=["svc", "st"]).astype(np.int8)
X_all = X_all.drop(columns=CATCOLS).join(oh)
X_all["proto_code"] = raw.proto.astype("category").cat.codes.astype(np.int16)
print("encoded: service + state one-hot (%d cols), proto ordinal" % oh.shape[1])

# --- dedupe on features + label, exactly as the CIC pipeline does ---
comb = pd.concat([X_all, y_all.rename("y")], axis=1).drop_duplicates().reset_index(drop=True)
X_all, y = comb.drop(columns="y"), comb["y"]
print("\nafter dedupe: %d rows (%.1f%% of pooled were duplicates)"
      % (len(comb), 100 * (1 - len(comb) / len(raw))))
print("CIC-2024 for reference: 447,915 -> 306,173 (32%% duplicates)")

dist = y.value_counts().sort_index()
print("\nclass distribution after dedupe")
for k, v in dist.items():
    print("  %-15s %7d  %5.2f%%" % (NAMES[k], v, 100 * v / len(y)))

pooled the official train/test files: 257673 rows x 45 cols
dropped: id, attack_cat (-> y), label (binary attack flag - would leak the gate's target)
cardinality: {'proto': 133, 'service': 13, 'state': 11}
encoded: service + state one-hot (24 cols), proto ordinal



after dedupe: 162745 rows (36.8% of pooled were duplicates)
CIC-2024 for reference: 447,915 -> 306,173 (32%% duplicates)

class distribution after dedupe
  Benign            85722  52.67%
  Analysis           2032   1.25%
  Backdoor           1880   1.16%
  DoS                5500   3.38%
  Exploits          27434  16.86%
  Fuzzers           20960  12.88%
  Generic            7599   4.67%
  Reconnaissance     9991   6.14%
  Shellcode          1456   0.89%
  Worms               171   0.11%


In [2]:
# --- unsupervised prune: identical rule to every CIC notebook (label-free, precedes the split) ---
nun = X_all.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = X_all.drop(columns=zero)
corr = FULL.corr(numeric_only=True)
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)
print("zero-variance dropped (%d): %s" % (len(zero), ", ".join(zero) or "none"))
print("|r|>=0.95 dropped  (%d): %s" % (len(red), ", ".join(red) or "none"))
print("FULL %d -> BASE %d features   (CIC was 67 -> 44)" % (FULL.shape[1], BASE.shape[1]))

zero-variance dropped (0): none
|r|>=0.95 dropped  (7): sbytes, dbytes, sloss, dloss, dwin, ct_ftp_cmd, st_FIN
FULL 64 -> BASE 57 features   (CIC was 67 -> 44)


### 1.1 Derived features — the UNSW analogue of round 1

Round 1 on CIC engineered 31 scale-free columns for +0.0101. **Most of them already exist natively
here** — `smean`/`dmean` are bytes-per-packet, `rate`/`sload`/`dload` are duration-normalised,
`sinpkt`/`dinpkt` are mean IATs, `sjit`/`djit` are jitter. That absence-of-need is itself a finding
and is reported below.

What remains constructible are the *asymmetries and dispersions* between the two directions, plus
ratios over the host-context counters, which have no CIC counterpart at all.

In [3]:
def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

def build_derived(F):
    d = {}
    g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
    sp, dp = g("spkts"), g("dpkts")
    sb, db = g("sbytes"), g("dbytes")
    tot_p, tot_b = sp + dp, sb + db

    # directional asymmetry (the CIC round-1 forms that DO have an analogue)
    d["r_pkt_sd"]  = safe(sp, dp);      d["r_byte_sd"] = safe(sb, db)
    d["asym_pkt"]  = safe(sp - dp, tot_p); d["asym_byte"] = safe(sb - db, tot_b)
    d["bpp_flow"]  = safe(tot_b, tot_p)
    d["r_mean_sd"] = safe(g("smean"), g("dmean"))
    d["r_load_sd"] = safe(g("sload"), g("dload"))
    d["r_inpkt_sd"] = safe(g("sinpkt"), g("dinpkt"))
    d["r_jit_sd"]  = safe(g("sjit"), g("djit"))

    # loss and jitter as RATES rather than counts
    d["loss_rate_s"] = safe(g("sloss"), sp); d["loss_rate_d"] = safe(g("dloss"), dp)
    d["cv_jit_s"] = safe(g("sjit"), g("sinpkt")); d["cv_jit_d"] = safe(g("djit"), g("dinpkt"))

    # TTL: the single most discriminative raw field in UNSW-NB15 in most published analyses
    d["ttl_diff"] = g("sttl") - g("dttl"); d["r_ttl"] = safe(g("sttl"), g("dttl"))

    # TCP handshake timing shares
    rtt = g("tcprtt")
    d["synack_share"] = safe(g("synack"), rtt); d["ackdat_share"] = safe(g("ackdat"), rtt)
    d["r_win_sd"] = safe(g("swin"), g("dwin"))

    # duration-normalised
    dur = g("dur")
    d["dur_per_pkt"] = safe(dur, tot_p); d["rate_per_pkt"] = safe(g("rate"), tot_p)
    d["resp_share"]  = safe(g("response_body_len"), db)

    # HOST-CONTEXT ratios - no CIC counterpart exists for these at all
    d["r_srv_src_ltm"] = safe(g("ct_srv_src"), g("ct_src_ltm"))
    d["r_srv_dst_ltm"] = safe(g("ct_srv_dst"), g("ct_dst_ltm"))
    d["r_dst_src_ltm"] = safe(g("ct_dst_src_ltm"), g("ct_src_ltm"))
    d["ctx_total"] = (g("ct_srv_src") + g("ct_dst_ltm") + g("ct_src_ltm") + g("ct_srv_dst"))
    d["r_sport_dport"] = safe(g("ct_src_dport_ltm"), g("ct_dst_sport_ltm"))

    O = pd.DataFrame(d, index=F.index).astype(np.float64)
    return O.replace([np.inf, -np.inf], 0.0).fillna(0.0)

DER = build_derived(FULL)
dz = DER.nunique(); DER = DER.drop(columns=dz[dz <= 1].index.tolist())
X = pd.concat([BASE, DER], axis=1)
DERIVED_NAMES = set(DER.columns)
CTX = [c for c in X.columns if c.startswith("ct_") or c in
       ("r_srv_src_ltm","r_srv_dst_ltm","r_dst_src_ltm","ctx_total","r_sport_dport")]
print("derived kept: %d -> DERIVED space %d features (CIC's was 75)" % (DER.shape[1], X.shape[1]))
print("host-context features available here: %d  (CIC has 0)" % len(CTX))
print(" ", ", ".join(CTX))

derived kept: 26 -> DERIVED space 83 features (CIC's was 75)
host-context features available here: 14  (CIC has 0)
  ct_srv_src, ct_state_ttl, ct_dst_ltm, ct_src_dport_ltm, ct_dst_sport_ltm, ct_dst_src_ltm, ct_flw_http_mthd, ct_src_ltm, ct_srv_dst, r_srv_src_ltm, r_srv_dst_ltm, r_dst_src_ltm, ctx_total, r_sport_dport


In [4]:
X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)
y_arr = y_trf.values

attack_labels = sorted(l for l in y.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels); FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values
y_te = y_test.values
att_te = y_te != 0
fam_te = pd.Series(y_te[att_te]).map(fam_to_idx).values
print("train+val %d (%d attack) | test %d (%d attack) | %d families"
      % (len(y_arr), att_trf.sum(), len(y_te), att_te.sum(), NC))

train+val 130196 (61618 attack) | test 32549 (15405 attack) | 9 families


## 2. The representation diagnostic — the point of the experiment

Same measurement as `knn_class_overlap.ipynb` and `feature_engineering.ipynb`: 1-NN error of test
attack rows against train attack rows, RobustScaler, Stage-B 9-family scope.

**One correction is needed to make the comparison fair.** The 1-NN error falls as the reference set
grows, and the two datasets have different attack counts, so the raw brackets are not comparable. The
UNSW reference set is therefore **subsampled to exactly the CIC attack-row count (64,936)**, and both
numbers are reported.

Per `feature_engineering_v2.ipynb`, the bracket is **metric-relative** and does not bound a tree
ensemble. It is used here for what it is valid for: comparing the *geometry* of two feature spaces
under one fixed metric.

In [5]:
def cover_hart(e1nn, c):
    inner = 1 - (c / (c - 1)) * e1nn
    return np.nan if inner < 0 else (c - 1) / c * (1 - np.sqrt(inner))

Xa_tr = X_trf[att_trf]; Xa_te = X_test[att_te]
sc = RobustScaler().fit(Xa_tr)
Ztr, Zte = sc.transform(Xa_tr), sc.transform(Xa_te)
fam_tr_a = fam_trf[att_trf]

rows = []
CIC_N = 64936   # CIC train+val attack rows, from feature_engineering.ipynb
for tag, n_ref in (("UNSW (all %d ref)" % len(Ztr), len(Ztr)),
                   ("UNSW (size-matched to CIC)", min(CIC_N, len(Ztr)))):
    rng = np.random.RandomState(RS)
    sel = rng.choice(len(Ztr), n_ref, replace=False) if n_ref < len(Ztr) else np.arange(len(Ztr))
    nn = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(Ztr[sel])
    _, ii = nn.kneighbors(Zte)
    e1 = float((fam_tr_a[sel][ii[:, 0]] != fam_te).mean())
    rows.append(dict(space=tag, n_ref=n_ref, e_1NN=round(e1, 4),
                     e_star_lo=round(cover_hart(e1, NC), 4), e_star_hi=round(e1, 4),
                     acc_ceil_lo=round(1 - e1, 4), acc_ceil_hi=round(1 - cover_hart(e1, NC), 4)))
rows.append(dict(space="CIC BASE 44 (published)", n_ref=CIC_N, e_1NN=0.3757,
                 e_star_lo=0.2135, e_star_hi=0.3757, acc_ceil_lo=0.6243, acc_ceil_hi=0.7865))
rows.append(dict(space="CIC DERIVED 75 (published)", n_ref=CIC_N, e_1NN=0.3629,
                 e_star_lo=0.2053, e_star_hi=0.3632, acc_ceil_lo=0.6368, acc_ceil_hi=0.7947))
print(pd.DataFrame(rows).to_string(index=False))

                     space  n_ref  e_1NN  e_star_lo  e_star_hi  acc_ceil_lo  acc_ceil_hi
      UNSW (all 61618 ref)  61618 0.3043     0.1680     0.3043       0.6957       0.8320
UNSW (size-matched to CIC)  61618 0.3043     0.1680     0.3043       0.6957       0.8320
   CIC BASE 44 (published)  64936 0.3757     0.2135     0.3757       0.6243       0.7865
CIC DERIVED 75 (published)  64936 0.3629     0.2053     0.3632       0.6368       0.7947


In [6]:
K = 10
nn = NearestNeighbors(n_neighbors=K, algorithm="brute", n_jobs=-1).fit(Ztr)
_, ii = nn.kneighbors(Zte)
own = (fam_tr_a[ii] == fam_te[:, None]).mean(1)
pur = pd.Series(own).groupby(fam_te).mean()
CIC_PUR = {"Analysis":0.2987,"Backdoor":0.2267,"DoS":0.2204,"Exploits":0.6797,"Fuzzers":0.6728,
           "Generic":0.4321,"Reconnaissance":0.5665,"Shellcode":0.1403,"Worms":0.1479}
P = pd.DataFrame({"UNSW": pur.values}, index=[FAM[i] for i in pur.index])
P["CIC_DERIVED"] = [CIC_PUR[n] for n in P.index]
P["delta"] = (P["UNSW"] - P["CIC_DERIVED"]).round(4)
P["support"] = pd.Series(fam_te).value_counts().sort_index().values
print("own-class neighbourhood purity at k=%d" % K)
print(P.round(4).to_string())
print("\noverall  UNSW %.4f   vs   CIC DERIVED 0.3761" % pur.mean())
print("\nAttractor check - CIC found Exploits and Fuzzers swallow the small families.")
for a, b in (("Analysis","Exploits"), ("Shellcode","Fuzzers"), ("DoS","Exploits"), ("Worms","Exploits")):
    ia, ib = FAM.index(a), FAM.index(b)
    m = fam_te == ia
    print("  %-15s neighbourhood is %5.1f%% %-10s (CIC: Analysis 51.4%%, Shellcode 46.5%%, "
          "DoS 43.5%%, Worms 43.4%%)" % (a, 100 * float((fam_tr_a[ii[m]] == ib).mean()), b))

own-class neighbourhood purity at k=10
                  UNSW  CIC_DERIVED   delta  support
Analysis        0.1843       0.2987 -0.1144      407
Backdoor        0.1016       0.2267 -0.1251      376
DoS             0.2356       0.2204  0.0152     1100
Exploits        0.7335       0.6797  0.0538     5487
Fuzzers         0.7973       0.6728  0.1245     4192
Generic         0.8237       0.4321  0.3916     1520
Reconnaissance  0.6215       0.5665  0.0550     1998
Shellcode       0.3543       0.1403  0.2140      291
Worms           0.0882       0.1479 -0.0597       34

overall  UNSW 0.4378   vs   CIC DERIVED 0.3761

Attractor check - CIC found Exploits and Fuzzers swallow the small families.
  Analysis        neighbourhood is  22.1% Exploits   (CIC: Analysis 51.4%, Shellcode 46.5%, DoS 43.5%, Worms 43.4%)
  Shellcode       neighbourhood is  15.5% Fuzzers    (CIC: Analysis 51.4%, Shellcode 46.5%, DoS 43.5%, Worms 43.4%)
  DoS             neighbourhood is  37.8% Exploits   (CIC: Analysis 51.4%

## 3. The v6 architecture, unchanged

Same 7-member alpha-diverse pool, same 5-fold cross-fit, same per-class $W$ by coordinate ascent,
same decision-space competence combiner. Only the data underneath is different.

In [7]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw); return XGBClassifier(**p)

wc_alpha = lambda n, a: ((n.sum() / n) ** a) / (((n.sum() / n) ** a).mean())
def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

SPECS = ([(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS); I25 = MODELS.index("XGB a=0.25")

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    m = make_xgb(); m.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb)); return m

CACHE = Path("cache_oof_unsw.npz")
if CACHE.exists():
    z = np.load(CACHE); OOF = [z[f"m{m}"] for m in range(M)]; oof_gate = z["gate"]
    print("loaded", CACHE)
else:
    OOF = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]
    oof_gate = np.zeros(len(y_arr), int)
    t0 = time.time()
    for f_, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr)):
        tf = time.time()
        oof_gate[va] = fit_gate(X_trf.iloc[tr], y_arr[tr]).predict(X_trf.iloc[va])
        ma = y_arr[tr] != 0
        Xa = X_trf.iloc[tr][ma].reset_index(drop=True); ya = pd.Series(fam_trf[tr][ma])
        for mi, (_, (k, p)) in enumerate(SPECS):
            OOF[mi][va] = fit_member(k, p, Xa, ya).predict_proba(X_trf.iloc[va]).astype(np.float32)
        print("  fold %d/5 (%.0fs)" % (f_ + 1, time.time() - tf))
    print("cross-fit %.0fs" % (time.time() - t0))
    np.savez_compressed(CACHE, gate=oof_gate, **{f"m{m}": OOF[m] for m in range(M)})

yb_tr = (y_arr != 0).astype(int)
print("OOF gate: attack recall %.4f / precision %.4f   (CIC v6 gate: 0.9961 / 0.9280)"
      % (recall_score(yb_tr, oof_gate), precision_score(yb_tr, oof_gate)))

loaded cache_oof_unsw.npz
OOF gate: attack recall 0.9176 / precision 0.9209   (CIC v6 gate: 0.9961 / 0.9280)


In [8]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
ref_idx = comb_idx[att_trf[comb_idx]]
y_ref = fam_trf[ref_idx]
CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])
WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)
prof = lambda ps: np.hstack(ps).astype(np.float32)
NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
    prof([OOF[m][ref_idx] for m in range(M)]))

def comp_ola(idx, k):
    nb = idx[:, :k]; wn = WC_REF[y_ref[nb]]; den = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M): W[:, m] = (CORRECT[m][nb] * wn).sum(1) / den
    return W

def fuse_static(probas, Wc):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M): S += probas[m] * Wc[m][None, :]
    return S.argmax(1)

def fuse_dyn(probas, Wd, p=1.0, Wclass=None):
    Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        coef = Wp[:, m:m+1] if Wclass is None else Wp[:, m:m+1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s > best + 1e-7: best, cur = s, gg
                W[m_, c] = cur
    return W

t0 = time.time()
W_class = fit_W([OOF[m][ref_idx] for m in range(M)], y_ref)
print("W fitted in %.0fs | reference set %d attack rows" % (time.time() - t0, len(ref_idx)))

W fitted in 16s | reference set 30809 attack rows


### 3.1 SELV selection, then the test set once

The combiner variant is chosen on SELV exactly as v6 did — the winner there is what gets quoted.

In [9]:
def score10(fam_g, ye, gmask, tag):
    yp = np.zeros(len(ye), int)
    if len(fam_g): yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_g)
    return dict(variant=tag,
                mF1=round(f1_score(ye, yp, average="macro", zero_division=0), 4),
                mP=round(precision_score(ye, yp, average="macro", zero_division=0), 4),
                mR=round(recall_score(ye, yp, average="macro", zero_division=0), 4),
                acc=round(float((yp == ye).mean()), 4))

def run_all(P_g, idx, ye, gmask):
    out = [score10(fuse_static(P_g, W_class), ye, gmask, "STATIC: per-class W"),
           score10(fuse_static(P_g, np.ones((M, NC)) / M), ye, gmask, "STATIC: equal average")]
    for m in range(M):
        out.append(score10(P_g[m].argmax(1), ye, gmask, "single: " + MODELS[m]))
    for k in (30, 100):
        for p in (1, 3, 8):
            out.append(score10(fuse_dyn(P_g, comp_ola(idx, k), p=p), ye, gmask,
                               f"DYN OLA-cw k={k} p={p}"))
    for k in (30, 100):
        out.append(score10(fuse_dyn(P_g, comp_ola(idx, k), p=8, Wclass=W_class), ye, gmask,
                           f"HYBRID W x OLA-cw k={k} p=8"))
    return pd.DataFrame(out)

gmask_selv = oof_gate[selv_idx] == 1
gpos = selv_idx[gmask_selv]
P_selv = [OOF[m][gpos] for m in range(M)]
_, i_selv = NN_D.kneighbors(prof(P_selv))
selv_tab = run_all(P_selv, i_selv, y_arr[selv_idx], gmask_selv).sort_values(
    "mF1", ascending=False).reset_index(drop=True)
print(selv_tab.to_string(index=False))
BESTV = selv_tab.iloc[0]["variant"]
print("\nSELV winner: %s (%.4f)   |  v6 on CIC chose 'DYN OLA-cw k=30 p=8'" % (BESTV, selv_tab.iloc[0]["mF1"]))

                    variant    mF1     mP     mR    acc
        STATIC: per-class W 0.6313 0.6898 0.6629 0.8258
HYBRID W x OLA-cw k=100 p=8 0.6300 0.6739 0.6605 0.8263
 HYBRID W x OLA-cw k=30 p=8 0.6299 0.6880 0.6617 0.8263
          single: XGB a=1.0 0.6005 0.5959 0.6212 0.8175
        DYN OLA-cw k=30 p=8 0.5979 0.6041 0.6073 0.8211
       DYN OLA-cw k=100 p=8 0.5961 0.6033 0.6032 0.8211
        DYN OLA-cw k=30 p=3 0.5948 0.6036 0.6013 0.8211
       DYN OLA-cw k=100 p=3 0.5913 0.6021 0.5942 0.8210
        DYN OLA-cw k=30 p=1 0.5900 0.6032 0.5909 0.8208
       DYN OLA-cw k=100 p=1 0.5877 0.6008 0.5876 0.8206
          single: XGB a=0.5 0.5847 0.5946 0.5863 0.8192
         single: XGB a=0.25 0.5802 0.5951 0.5759 0.8187
      STATIC: equal average 0.5790 0.5917 0.5760 0.8191
          single: RF a=0.25 0.5711 0.5851 0.5667 0.8170
          single: XGB a=0.0 0.5648 0.5830 0.5545 0.8179
      single: XGB smote2000 0.5641 0.5778 0.5590 0.8177
          single: ET a=0.25 0.5474 0.5836 0.5282

In [10]:
t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa_f = X_trf[att_trf].reset_index(drop=True); ya_f = pd.Series(fam_trf[att_trf])
FINAL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
gmask_test = gate_f.predict(X_test) == 1
P_test = [m.predict_proba(X_test.iloc[np.where(gmask_test)[0]]).astype(np.float32) for m in FINAL]
_, i_test = NN_D.kneighbors(prof(P_test))
print("refit + test in %.0fs" % (time.time() - t0))

test_tab = run_all(P_test, i_test, y_te, gmask_test).sort_values(
    "mF1", ascending=False).reset_index(drop=True)
print("\n--- TEST (selection already made on SELV) ---")
print(test_tab.to_string(index=False))
sel = test_tab[test_tab.variant == BESTV].iloc[0]
print("\nSELV-selected '%s' ranks %d of %d on test"
      % (BESTV, 1 + list(test_tab.variant).index(BESTV), len(test_tab)))
print("UNSW-NB15: macro-F1 %.4f | accuracy %.4f" % (sel["mF1"], sel["acc"]))
print("CIC-2024 v6 reference: macro-F1 0.6485 | accuracy 0.9123")
print("\n!! the two are NOT directly comparable in level - different class balance")
print("   (Normal 36%% here vs Benign 73.5%% there) and different row counts.")
print("   The per-family and purity comparisons below are the meaningful ones.")

refit + test in 111s



--- TEST (selection already made on SELV) ---
                    variant    mF1     mP     mR    acc
HYBRID W x OLA-cw k=100 p=8 0.6394 0.7170 0.6723 0.8312
        STATIC: per-class W 0.6387 0.7237 0.6707 0.8303
 HYBRID W x OLA-cw k=30 p=8 0.6349 0.7200 0.6678 0.8302
        DYN OLA-cw k=30 p=1 0.6082 0.6295 0.6040 0.8251
          single: XGB a=1.0 0.6079 0.6111 0.6209 0.8210
       DYN OLA-cw k=100 p=3 0.6079 0.6252 0.6074 0.8254
        DYN OLA-cw k=30 p=8 0.6078 0.6233 0.6085 0.8250
        DYN OLA-cw k=30 p=3 0.6074 0.6249 0.6068 0.8254
       DYN OLA-cw k=100 p=8 0.6067 0.6205 0.6092 0.8254
       DYN OLA-cw k=100 p=1 0.6066 0.6321 0.5989 0.8251
      STATIC: equal average 0.5978 0.6233 0.5859 0.8235
          single: XGB a=0.5 0.5948 0.6092 0.5941 0.8226
         single: XGB a=0.25 0.5936 0.6205 0.5811 0.8235
          single: RF a=0.25 0.5929 0.6150 0.5832 0.8228
      single: XGB smote2000 0.5850 0.6122 0.5721 0.8233
          single: XGB a=0.0 0.5670 0.6029 0.5482 0.8228
 

## 4. The comparison that matters — same traffic, same taxonomy, different extractor

In [11]:
def build(t, P_g, idx):
    if t == "STATIC: per-class W":   return fuse_static(P_g, W_class)
    if t == "STATIC: equal average": return fuse_static(P_g, np.ones((M, NC)) / M)
    if t.startswith("single: "):     return P_g[MODELS.index(t[8:])].argmax(1)
    k = 100 if "k=100" in t else 30
    p = 8 if "p=8" in t else (3 if "p=3" in t else 1)
    return fuse_dyn(P_g, comp_ola(idx, k), p=p,
                    Wclass=(W_class if t.startswith("HYBRID") else None))

fam_pred = build(BESTV, P_test, i_test)
yp = np.zeros(len(y_te), int)
yp[gmask_test] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred)
print(classification_report(y_te, yp, labels=list(range(10)), target_names=NAMES,
                            digits=3, zero_division=0))

# v6 per-class F1, read from final_model_v6.ipynb's own classification_report
V6 = dict(zip(NAMES, [0.985, 0.446, 0.635, 0.468, 0.791, 0.721, 0.828, 0.640, 0.388, 0.582]))
f1u = f1_score(y_te, yp, average=None, labels=range(10), zero_division=0)
sup = pd.Series(y_te).value_counts().sort_index()
cmp_ = pd.DataFrame({"UNSW_F1": f1u.round(3), "CIC_v6_F1": [V6[n] for n in NAMES],
                     "delta": (f1u - np.array([V6[n] for n in NAMES])).round(3),
                     "UNSW_support": sup.reindex(range(10)).fillna(0).astype(int).values},
                    index=NAMES)
print("\nper-family F1: same families, same traffic, different feature extractor")
print(cmp_.to_string())
print("\nCIC's four worst families were Shellcode .388, Analysis .446, DoS .468, Worms .582")
print("If those move up here, the plateau was the representation. If not, it is the traffic.")

                precision    recall  f1-score   support

        Benign      0.927     0.932     0.930     17144
      Analysis      0.163     0.779     0.270       407
      Backdoor      0.974     0.101     0.183       376
           DoS      0.472     0.342     0.396      1100
      Exploits      0.876     0.822     0.849      5487
       Fuzzers      0.702     0.648     0.674      4192
       Generic      0.953     0.896     0.924      1520
Reconnaissance      0.883     0.742     0.807      1998
     Shellcode      0.580     0.739     0.650       291
         Worms      0.706     0.706     0.706        34

      accuracy                          0.830     32549
     macro avg      0.724     0.671     0.639     32549
  weighted avg      0.860     0.830     0.838     32549


per-family F1: same families, same traffic, different feature extractor
                UNSW_F1  CIC_v6_F1  delta  UNSW_support
Benign            0.930      0.985 -0.055         17144
Analysis          0.270     

In [12]:
imp = pd.Series(FINAL[I25].feature_importances_, index=X.columns).sort_values(ascending=False)
tag = lambda k: "(ctx)" if k in CTX else ("(der)" if k in DERIVED_NAMES else "")
print("top 20 Stage-B features")
for i, (k, v) in enumerate(imp.head(20).items()):
    print("  %2d. %-22s %.4f %s" % (i + 1, k, v, tag(k)))
ctx_share = imp[[c for c in imp.index if c in CTX]].sum() / imp.sum()
der_share = imp[[c for c in imp.index if c in DERIVED_NAMES]].sum() / imp.sum()
print("\nHOST-CONTEXT features: %d of %d columns (%.1f%%), carrying %.1f%% of importance"
      % (len(CTX), X.shape[1], 100 * len(CTX) / X.shape[1], 100 * ctx_share))
print("derived features    : %.1f%% of importance" % (100 * der_share))
print("\nThis is the number to quote: CICFlowMeter cannot produce ANY of the context features,")
print("and feature_engineering_v2.ipynb proved they cannot be reconstructed from that file.")

top 20 Stage-B features
   1. svc_dns                0.2950 
   2. ttl_diff               0.2259 (der)
   3. svc_http               0.0528 
   4. sttl                   0.0308 
   5. svc_-                  0.0282 
   6. r_ttl                  0.0215 (der)
   7. bpp_flow               0.0208 (der)
   8. svc_ftp-data           0.0195 
   9. r_byte_sd              0.0155 (der)
  10. svc_irc                0.0153 
  11. ct_srv_dst             0.0152 (ctx)
  12. svc_snmp               0.0145 
  13. dmean                  0.0123 
  14. r_sport_dport          0.0120 (ctx)
  15. r_mean_sd              0.0119 (der)
  16. svc_ftp                0.0116 
  17. svc_smtp               0.0104 
  18. ct_dst_sport_ltm       0.0101 (ctx)
  19. smean                  0.0088 
  20. loss_rate_d            0.0087 (der)

HOST-CONTEXT features: 14 of 83 columns (16.9%), carrying 6.8% of importance
derived features    : 37.7% of importance

This is the number to quote: CICFlowMeter cannot produce ANY of the co

## 5. Findings

**The central claim survives, but the mechanism is not the one the project predicted.** Holding the
traffic and the taxonomy fixed and swapping only the feature extractor moves the geometry
substantially — so the plateau *is* representation-bound. But the columns doing the work are **not**
the host/temporal context this project spent two notebooks arguing for.

### 1. The representation is genuinely more separable

| Space | $n_{ref}$ | $e_{1NN}$ | $e^\ast$ bracket | accuracy ceiling |
|---|---|---|---|---|
| CIC BASE 44 | 64,936 | 0.3757 | [0.2135, 0.3757] | [0.6243, 0.7865] |
| CIC DERIVED 75 | 64,936 | 0.3629 | [0.2053, 0.3632] | [0.6368, 0.7947] |
| **UNSW-NB15 83** | **61,618** | **0.3043** | **[0.1680, 0.3043]** | **[0.6957, 0.8320]** |

$e_{1NN}$ falls by **0.0586** against CIC DERIVED — **4.6x the entire round-1 feature-engineering
gain** (−0.0128) and larger than round 1 and round 2 combined. The whole bracket shifts up by ~6
accuracy points.

**The comparison is conservative.** The size-matching step was a no-op: UNSW has *fewer* attack rows
(61,618) than CIC (64,936), and a smaller reference set can only *raise* the 1-NN error. So the UNSW
advantage is if anything understated.

Read this with the metric caveat from `feature_engineering_v2.ipynb` — the bracket is
metric-relative and does not bound a tree ensemble. Its legitimate use is exactly this one: comparing
the geometry of two feature spaces under one fixed metric. Here it is not a rescaling of the same
information; it is a different measurement of the same traffic.

### 2. THE SURPRISE — the missing ingredient was not host/temporal context

`knn_class_overlap.ipynb` concluded the way forward was "payload, **host/temporal context**, session
aggregation", and `feature_engineering_v2.ipynb` proved such features were impossible on the CIC file.
UNSW-NB15 has 14 of them. **They carry only 6.8% of the model's importance.**

What actually dominates:

| rank | feature | importance | in CICFlowMeter? |
|---|---|---|---|
| 1 | `svc_dns` | **0.295** | **no — no service/port column at all** |
| 2 | `ttl_diff` (derived) | **0.226** | **no — no TTL column at all** |
| 3 | `svc_http` | 0.053 | no |
| 4 | `sttl` | 0.031 | no |
| 6 | `r_ttl` (derived) | 0.022 | no |

**The top five features are all built from two fields CICFlowMeter does not record: service identity
and TTL.** Service one-hots plus TTL-derived columns account for well over half the model's
importance, while the connection-history counters the project hypothesised about contribute almost
nothing.

This is a sharper and more actionable conclusion than the one it replaces: the CIC representation's
deficit is **protocol/service identity and TTL**, not temporal context. It also explains why two
rounds of algebra on the CIC columns returned nothing — no ratio of packet counts and durations can
reconstruct "this flow is DNS" or "the TTL is 254 in one direction and 62 in the other".

### 3. Performance mostly tracks purity — but the rare families split two ways

| Family | purity Δ (UNSW − CIC) | F1 Δ (UNSW − CIC v6) | support | consistent? |
|---|---|---|---|---|
| Generic | **+0.392** | +0.096 | 1,520 | yes |
| Shellcode | **+0.214** | **+0.262** | 291 | yes |
| Fuzzers | +0.125 | −0.047 | 4,192 | **no** |
| Reconnaissance | +0.055 | **+0.167** | 1,998 | yes |
| Exploits | +0.054 | +0.058 | 5,487 | yes |
| DoS | +0.015 | −0.072 | 1,100 | **no** |
| Worms | −0.060 | +0.124 | **34** | **no** |
| Analysis | **−0.114** | **−0.176** | 407 | yes |
| Backdoor | **−0.125** | **−0.452** | 376 | yes |

Purity and F1 move together for **6 of 9** families, including both directions of change — the two
whose neighbourhoods degrade (Analysis, Backdoor) are also the two whose F1 collapses. The three
exceptions are Fuzzers and DoS (small purity gains that did not convert) and Worms, whose **34 test
rows** make its number meaningless in either direction.

**The clearest single win is Shellcode: 0.388 → 0.650**, and it is exactly what the CIC overlap
analysis predicted would happen if the representation improved — Shellcode's neighbourhood is
**15.5% Fuzzers here versus 46.5% on CIC**, so the attractor that swallowed it largely dissolves.
Reconnaissance (0.640 → 0.807) and Generic (0.828 → 0.924) follow the same pattern.

**But the improvement is not uniform across the weak families, and this must be stated plainly.** Of
CIC's four weakest families — Shellcode 0.388, Analysis 0.446, DoS 0.468, Worms 0.582 — only
**Shellcode clearly improves**; Analysis and DoS get *worse*, and Worms is noise. The representation
change is not a rising tide: it trades one set of confusions for another.

**Two families collapse.** **Backdoor 0.635 → 0.183** is the worst single result in the project:
precision 0.974, recall **0.101** — the model is nearly always right when it says Backdoor, and
nearly never says it. Analysis is the mirror image (precision 0.163, recall 0.779). The two have
degenerated into a swapped pair, a known pathology of UNSW-NB15's Analysis/Backdoor/DoS labelling
rather than something this pipeline introduced.

### 4. The gate inverts — CIC is better at the binary task

| | attack recall | attack precision |
|---|---|---|
| CIC v6 gate | **0.9961** | 0.9280 |
| UNSW gate | 0.9176 | 0.9209 |

On CIC the gate is "essentially solved" and Stage B is the sole bottleneck. Here it **misses 8% of
attacks** — an entirely different error budget, and the reason end-to-end accuracy is lower despite
the better family geometry. **The two representations are good at different halves of the problem**:
CICFlowMeter's timing/size statistics separate attack from benign; UNSW's service and TTL fields
separate family from family.

That is a genuinely useful result for the write-up, and it suggests the strongest available system is
neither dataset alone but the union of the two field sets — which is exactly what a re-extraction with
both tools would produce, and is out of scope here.

### 5. The static per-class $W$ beats every dynamic combiner — v4's finding, replicated

SELV chose `STATIC: per-class W`; on test it ranks 2 of 17 at 0.6387, behind
`HYBRID W x OLA-cw k=100 p=8` at 0.6394 — a gap of **0.0007**, inside any noise band, and the top
three variants all carry the per-class $W$.

The pure-dynamic combiners cluster far below at ~0.607, a **−0.031** deficit. This is a much sharper
version of what v4 found on CIC ("the per-class prior is load-bearing; the per-instance term is
not"), now **replicated on an independent representation**. The ensembling gain over the single
α=0.25 model is **+0.045** here versus +0.023 on CIC.

Note also that on CIC, SELV picked a *dynamic* combiner and here it picks a *static* one — the
selection protocol adapts to the dataset rather than inheriting a fixed answer, which is the correct
behaviour.

### 6. What must NOT be claimed

- **This does not close the "single-dataset evaluation" gap.** Same source traffic, same taxonomy,
  same ten families — a different *extractor*, not different data. Closing that gap still requires
  genuinely different traffic (CIC-IDS2017 or similar).
- **The headline numbers are not comparable in level.** macro-F1 0.6387 vs 0.6485 and accuracy 0.8303
  vs 0.9123 sit on different class balances (Benign **52.7%** here after dedupe vs 73.5% on CIC),
  different row counts (162,745 vs 306,173 after dedupe), and different flow segmentation of the same
  traffic (257,673 vs 447,915 raw flows). Only the **per-family**, **purity** and **bracket**
  comparisons are meaningful.
- **The UNSW pre-split files are a curated subset**, not the full 2.5M-row release, so its class
  balance is itself an artifact of how that subset was drawn.

### 7. Where this leaves the thesis

The representation argument is now supported by evidence from *outside* the CIC feature space rather
than only from within it, which is much stronger — and it comes with a concrete, falsifiable
mechanism: **service identity and TTL**, two fields CICFlowMeter does not emit, carry the majority of
the discriminative power on this traffic. Future-work framing should name those specifically instead
of "richer features" generically.